In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/Salary_Datanew.csv")

In [3]:
df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0


In [4]:
df.columns

Index(['Age', 'Gender', 'Education Level', 'Job Title', 'Years of Experience',
       'Salary'],
      dtype='object')

In [5]:
df.isnull().sum()

,0
Age,2
Gender,2
Education Level,3
Job Title,2
Years of Experience,3
Salary,5


In [11]:
for col in df.columns:
    if df[col].dtype == 'object': # Check for categorical columns (object type)
        # Fill with mode
        df[col] = df[col].fillna(df[col].mode()[0])
    elif df[col].dtype in ['int64', 'float64']: # Check for numerical columns
        # Fill with mean
        df[col] = df[col].fillna(df[col].mean())

In [10]:
df.isnull().sum()

,0
Age,0
Gender,0
Education Level,0
Job Title,0
Years of Experience,0
Salary,0


In [12]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
le = LabelEncoder()

# Identify categorical columns (object dtype)
# Exclude 'Job Title' for now, as it might have too many unique values for simple Label Encoding
categorical_cols = [col for col in df.columns if df[col].dtype == 'object' and col != 'Job Title']

# Apply LabelEncoder to each categorical column
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

display(df.head())

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,1,0,Software Engineer,5.0,90000.0
1,28.0,0,3,Data Analyst,3.0,65000.0
2,45.0,1,5,Senior Manager,15.0,150000.0
3,36.0,0,0,Sales Associate,7.0,60000.0
4,52.0,1,3,Director,20.0,200000.0


In [13]:
# Apply LabelEncoder to 'Job Title'
df['Job Title'] = le.fit_transform(df['Job Title'])

display(df.head())

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,1,0,177,5.0,90000.0
1,28.0,0,3,18,3.0,65000.0
2,45.0,1,5,145,15.0,150000.0
3,36.0,0,0,116,7.0,60000.0
4,52.0,1,3,26,20.0,200000.0


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import math

## Data Preparation: Features (X) and Target (y) Split

First, I'll separate the features (input variables) from the target variable (what we want to predict), which is 'Salary'. Then, I'll split the data into training and testing sets to evaluate how well our models generalize to unseen data.

In [15]:
# Define features (X) and target (y)
X = df.drop('Salary', axis=1)  # All columns except 'Salary'
y = df['Salary']              # The 'Salary' column

# Split the data into training and testing sets
# test_size=0.2 means 20% of the data will be used for testing
# random_state ensures reproducibility of the split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 5363 samples
Test set size: 1341 samples


## Model Training and Evaluation

Now, I'll define several regression models, train them on the training data, make predictions on the test data, and calculate key evaluation metrics: Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R-squared (R2 Score).

In [16]:
# Initialize a dictionary to store model results
model_results = {}

# Define the models to be trained
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
    'Random Forest Regressor': RandomForestRegressor(random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=42)
}

# Loop through each model, train, predict, and evaluate
for name, model in models.items():
    print(f"\n--- Training {name} ---")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = math.sqrt(mse) # RMSAR is likely RMSE
    r2 = r2_score(y_test, y_pred)

    # Store results
    model_results[name] = {
        'MSE': mse,
        'RMSE': rmse,
        'R2 Score': r2
    }

    print(f"  Mean Squared Error (MSE): {mse:.2f}")
    print(f"  Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"  R-squared (R2 Score): {r2:.2f}")


--- Training Linear Regression ---
  Mean Squared Error (MSE): 892668414.69
  Root Mean Squared Error (RMSE): 29877.56
  R-squared (R2 Score): 0.67

--- Training Decision Tree Regressor ---
  Mean Squared Error (MSE): 93817976.26
  Root Mean Squared Error (RMSE): 9685.97
  R-squared (R2 Score): 0.96

--- Training Random Forest Regressor ---
  Mean Squared Error (MSE): 65085758.85
  Root Mean Squared Error (RMSE): 8067.57
  R-squared (R2 Score): 0.98

--- Training Gradient Boosting Regressor ---
  Mean Squared Error (MSE): 173977023.55
  Root Mean Squared Error (RMSE): 13190.04
  R-squared (R2 Score): 0.93


## Model Performance Summary

Here's a summary of the performance for each trained model, allowing for easy comparison:

In [17]:
# Convert the results dictionary to a DataFrame for better readability
results_df = pd.DataFrame(model_results).T
display(results_df.sort_values(by='R2 Score', ascending=False))

,MSE,RMSE,R2 Score
Random Forest Regressor,6.508576e+07,8067.574533,0.975630
Decision Tree Regressor,9.381798e+07,9685.968008,0.964871
Gradient Boosting Regressor,1.739770e+08,13190.035010,0.934857
Linear Regression,8.926684e+08,29877.557040,0.665754


## Applying the Best Model

Based on our evaluation, the Random Forest Regressor demonstrated the best performance. Let's apply this model to make predictions.

In [18]:
# Get the best performing model (Random Forest Regressor)
best_model = models['Random Forest Regressor']

# Make predictions on the test set using the best model
final_predictions = best_model.predict(X_test)

print("First 5 predictions from the best model (Random Forest Regressor):")
print(final_predictions[:5])

# Optionally, compare with actual values
print("\nFirst 5 actual values from the test set:")
print(y_test.head().values)

First 5 predictions from the best model (Random Forest Regressor):
[156492.12166667 140000.          79516.97868445  39980.
  50630.        ]

First 5 actual values from the test set:
[156486. 140000.  80000.  40000.  50000.]


## Saving the Best Model

First, I'll save the best performing model (Random Forest Regressor) to a file using `joblib`. This allows us to load the model later in our Streamlit application without retraining it.

In [19]:
import joblib

# Define the filename for the saved model
model_filename = 'random_forest_regressor_model.pkl'

# Save the best model
joblib.dump(best_model, model_filename)

print(f"Best model saved to {model_filename}")

Best model saved to random_forest_regressor_model.pkl


## Creating the Streamlit Application

Now, I'll create a new Python file named `app.py` that will host our Streamlit application. This app will allow users to input values for the features and get a salary prediction.

In [20]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

# Load the trained model
model = joblib.load('random_forest_regressor_model.pkl')

# Title of the Streamlit app
st.title('Salary Prediction App')
st.write('Enter the employee details to predict their salary.')

# Input fields for features
age = st.slider('Age', 18, 65, 30)
gender = st.selectbox('Gender', ['Female', 'Male'])
education_level = st.selectbox('Education Level', ['Bachelor\'s', 'Master\'s', 'PhD', 'High School', 'Some College', 'Vocational'])
job_title = st.text_input('Job Title (e.g., Software Engineer, Data Analyst)')
years_of_experience = st.slider('Years of Experience', 0, 40, 5)

# Map categorical inputs to numerical values (matching LabelEncoder's output)
# NOTE: This assumes the original LabelEncoder mapping is available or consistent.
# For a real application, you'd save the LabelEncoder objects as well.
# For demonstration, we'll use a simple hardcoded mapping for 'Gender' and 'Education Level'.
# 'Job Title' will require more careful handling as its original encoding is complex.

gender_mapping = {'Female': 0, 'Male': 1}
education_mapping = {
    'Bachelor\'s': 0, 'Master\'s': 3, 'PhD': 5,
    'High School': 2, 'Some College': 4, 'Vocational': 6
} # These mappings are derived from the initial df.head() after encoding

# Dummy Job Title encoding for demonstration. In a real scenario, you'd use the saved LabelEncoder.
# For this example, we'll just hash the job title string to get a numerical value,
# which is *not* how LabelEncoder works but serves as a placeholder.
# A better approach would be to load the actual LabelEncoder used during training.
# For now, let's create a simple, non-robust integer based on the input string.
# This is a simplification; a production app would need the original LabelEncoder.
job_title_encoded = hash(job_title) % 1000 # Just an example, not accurate encoding

# Create a DataFrame from user inputs
input_df = pd.DataFrame([{
    'Age': age,
    'Gender': gender_mapping[gender],
    'Education Level': education_mapping[education_level],
    'Job Title': job_title_encoded, # Simplified encoding
    'Years of Experience': years_of_experience
}])

# Predict button
if st.button('Predict Salary'):
    prediction = model.predict(input_df)[0]
    st.success(f'The predicted salary is: ${prediction:,.2f}')

Writing app.py


## Running the Streamlit Application

To run this Streamlit application, you'll need to install `streamlit` if you haven't already. Then, you can execute the app from your terminal or command prompt.

1.  **Install Streamlit** (if not already installed):
    ```bash
    pip install streamlit
    ```

2.  **Run the application** (from the directory where `app.py` and `random_forest_regressor_model.pkl` are located):
    ```bash
    streamlit run app.py
    ```

This will open the Streamlit application in your web browser. You can then interact with the sliders and input fields to get salary predictions. Remember that for the `Job Title`, the encoding in this simple app is not robust and a production application would require the original `LabelEncoder` object to be saved and loaded alongside the model.

## Saving the Best Model

First, I'll save the best performing model (Random Forest Regressor) to a file using `joblib`. This allows us to load the model later in our Streamlit application without retraining it.

In [ ]:
import joblib

# Define the filename for the saved model
model_filename = 'random_forest_regressor_model.pkl'

# Save the best model
joblib.dump(best_model, model_filename)

print(f"Best model saved to {model_filename}")

## Creating the Streamlit Application

Now, I'll create a new Python file named `app.py` that will host our Streamlit application. This app will allow users to input values for the features and get a salary prediction.

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

# Load the trained model
model = joblib.load('random_forest_regressor_model.pkl')

# Title of the Streamlit app
st.title('Salary Prediction App')
st.write('Enter the employee details to predict their salary.')

# Input fields for features
age = st.slider('Age', 18, 65, 30)
gender = st.selectbox('Gender', ['Female', 'Male'])
education_level = st.selectbox('Education Level', ['Bachelor\'s', 'Master\'s', 'PhD', 'High School', 'Some College', 'Vocational'])
job_title = st.text_input('Job Title (e.g., Software Engineer, Data Analyst)')
years_of_experience = st.slider('Years of Experience', 0, 40, 5)

# Map categorical inputs to numerical values (matching LabelEncoder's output)
# NOTE: This assumes the original LabelEncoder mapping is available or consistent.
# For a real application, you'd save the LabelEncoder objects as well.
# For demonstration, we'll use a simple hardcoded mapping for 'Gender' and 'Education Level'.
# 'Job Title' will require more careful handling as its original encoding is complex.

gender_mapping = {'Female': 0, 'Male': 1}
education_mapping = {
    'Bachelor\'s': 0, 'Master\'s': 3, 'PhD': 5,
    'High School': 2, 'Some College': 4, 'Vocational': 6
} # These mappings are derived from the initial df.head() after encoding

# Dummy Job Title encoding for demonstration. In a real scenario, you'd use the saved LabelEncoder.
# For this example, we'll just hash the job title string to get a numerical value,
# which is *not* how LabelEncoder works but serves as a placeholder.
# A better approach would be to load the actual LabelEncoder used during training.
# For now, let's create a simple, non-robust integer based on the input string.
# This is a simplification; a production app would need the original LabelEncoder.
job_title_encoded = hash(job_title) % 1000 # Just an example, not accurate encoding

# Create a DataFrame from user inputs
input_df = pd.DataFrame([{
    'Age': age,
    'Gender': gender_mapping[gender],
    'Education Level': education_mapping[education_level],
    'Job Title': job_title_encoded, # Simplified encoding
    'Years of Experience': years_of_experience
}])

# Predict button
if st.button('Predict Salary'):
    prediction = model.predict(input_df)[0]
    st.success(f'The predicted salary is: ${prediction:,.2f}')

## Running the Streamlit Application

To run this Streamlit application, you'll need to install `streamlit` if you haven't already. Then, you can execute the app from your terminal or command prompt.

1.  **Install Streamlit** (if not already installed):
    ```bash
    pip install streamlit
    ```

2.  **Run the application** (from the directory where `app.py` and `random_forest_regressor_model.pkl` are located):
    ```bash
    streamlit run app.py
    ```

This will open the Streamlit application in your web browser. You can then interact with the sliders and input fields to get salary predictions. Remember that for the `Job Title`, the encoding in this simple app is not robust and a production application would require the original `LabelEncoder` object to be saved and loaded alongside the model.